<a href="https://colab.research.google.com/github/ClarenceBallensky/MLP-vs-GCN-ids/blob/main/Build_Sample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Sample

## Imports

In [9]:
import os, zipfile, hashlib
import pandas as pd
import pyarrow.parquet as pq
from google.colab import drive, files

## Build Sample (maintainer use only)
Rebuilds ids_sample.parquet from the full improved CSE-CIC-IDS2018 download.
Needs a lot of RAM and time. Not required to run the main notebook.

In [10]:
drive.mount('/content/drive')

URL = "https://intrusion-detection.distrinet-research.be/CNS2022/Datasets/CSECICIDS2018_improved.zip"
FRAC = 0.10
TARGET_ROWS = 300_000
SEED = 42
OUT = "/content/drive/MyDrive/ids_samples"
os.makedirs(OUT, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1: download and sample 10% of each daily CSV (saved per file so a crash loses little)


In [11]:
!wget -q -c -O /content/data.zip {URL}

with zipfile.ZipFile("/content/data.zip") as z:
    for name in z.namelist():
        out = f"{OUT}/{os.path.basename(name)}.parquet"
        if name.endswith(".csv") and not os.path.exists(out):
            chunks = pd.read_csv(z.open(name), chunksize=200_000)
            sample = [c.dropna().sample(frac=FRAC, random_state=SEED) for c in chunks]
            pd.concat(sample).to_parquet(out)
            print("done:", name)

!ls -lh {OUT}

total 1.3G
-rw------- 1 root root 130M Nov 11 21:50 Friday-02-03-2018.csv.parquet
-rw------- 1 root root 137M Nov 11 21:41 Friday-16-02-2018.csv.parquet
-rw------- 1 root root 122M Nov 11 21:45 Friday-23-02-2018.csv.parquet
-rw------- 1 root root 135M Nov 11 21:48 Thursday-01-03-2018.csv.parquet
-rw------- 1 root root 106M Nov 11 21:49 Thursday-15-02-2018.csv.parquet
-rw------- 1 root root 124M Nov 11 21:44 Thursday-22-02-2018.csv.parquet
-rw------- 1 root root 120M Nov 11 21:42 Tuesday-20-02-2018.csv.parquet
-rw------- 1 root root 113M Nov 11 21:40 Wednesday-14-02-2018.csv.parquet
-rw------- 1 root root 134M Nov 11 21:43 Wednesday-21-02-2018.csv.parquet
-rw------- 1 root root 134M Nov 11 21:47 Wednesday-28-02-2018.csv.parquet


## Step 2: combine into one ~300k-row file and print its hash

In [12]:
parquet_files = sorted(f for f in os.listdir(OUT) if f.endswith(".parquet"))
total = sum(pq.ParquetFile(f"{OUT}/{f}").metadata.num_rows for f in parquet_files)
frac = min(1.0, TARGET_ROWS / total)
print(total, "rows available; keeping fraction", round(frac, 3))

df = pd.concat(
    pd.read_parquet(f"{OUT}/{f}").sample(frac=frac, random_state=SEED)
    for f in parquet_files
).reset_index(drop=True)

fl = df.select_dtypes("float64").columns
df[fl] = df[fl].astype("float32")

df.to_parquet("/content/ids_sample.parquet")
print(df.shape)
print(os.path.getsize("/content/ids_sample.parquet") / 1e6, "MB")
print("sha256:", hashlib.sha256(open("/content/ids_sample.parquet", "rb").read()).hexdigest())

files.download("/content/ids_sample.parquet")

6319514 rows available; keeping fraction 0.047
(300001, 91)
56.817488 MB
sha256: f4eb850ee32419da5ab3aee9b33da28b1b3a3ab175c7e7af64a4311163fb03d5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>